# PPO for Continuous Actions: Landing on the Moon

CSCI 6353 · Topic 40.

This is the **same PPO** as Topic 39 (clipped objective + GAE + K-epoch reuse). The only
change is the policy: a **Gaussian** over continuous actions instead of a softmax. The network
outputs a *mean* action $\mu$ and a learned *spread* $\sigma$; we sample $a\sim\mathcal N(\mu,\sigma^2)$.
We train it to land a rocket on **LunarLanderContinuous-v3** (8-dim state, 2 continuous engines).

Runs on **CPU** but is heavier than CartPole — a solved landing takes ~1M steps (tens of
minutes). The cell below trains a shorter run so you can watch it climb.

## 1. Setup  (LunarLander needs Box2D)

In [ ]:
!pip -q install swig
!pip -q install "gymnasium[box2d]" torch matplotlib

In [ ]:
import gymnasium as gym
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.distributions import Normal

LR, GAMMA, LMBDA, EPS_CLIP = 3e-4, 0.99, 0.95, 0.2
ROLLOUT, K_EPOCH, MINIBATCH = 2048, 10, 64

## 2. The Gaussian actor-critic

A shared body feeds a **`mu` head** (the mean action, 2 numbers) plus a learned **`log_std`**
parameter, and a critic **`v`**. `pi` returns a Gaussian instead of a softmax.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(8, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc_mu = nn.Linear(64, 2)                    # mean action
        self.log_std = nn.Parameter(torch.zeros(2))      # learned spread
        self.fc_v = nn.Linear(64, 1)                     # critic
    def _body(self, x):
        x = torch.tanh(self.fc1(x)); return torch.tanh(self.fc2(x))
    def pi(self, x):
        h = self._body(x)
        mu = self.fc_mu(h)
        return mu, torch.exp(self.log_std).expand_as(mu)
    def v(self, x):
        return self.fc_v(self._body(x)).squeeze(-1)

## 3. Train  (step-based PPO: collect a rollout, done-masked GAE, K-epoch clipped updates)

Every ~2048 steps we compute GAE advantages, normalize them, and take several clipped
minibatch updates. Lower `TOTAL_STEPS` to iterate faster; raise it toward 1,000,000 for a
clean landing (return past +200).

In [ ]:
TOTAL_STEPS = 300_000    # ~a few minutes; use ~1_000_000 to fully solve

env = gym.make('LunarLanderContinuous-v3')
net = ActorCritic(); opt = optim.Adam(net.parameters(), lr=LR)
s, _ = env.reset(seed=0)
ep_ret, recent, curve, gstep = 0.0, [], [], 0
while gstep < TOTAL_STEPS:
    S, A, R, LP, D, V = [], [], [], [], [], []
    for _ in range(ROLLOUT):
        st = torch.from_numpy(s).float()
        with torch.no_grad():
            mu, std = net.pi(st); dist = Normal(mu, std)
            a = dist.sample(); logp = dist.log_prob(a).sum().item(); v = net.v(st).item()
        s2, r, term, trunc, _ = env.step(torch.clamp(a, -1, 1).numpy())
        done = term or trunc
        S.append(s); A.append(a.numpy()); R.append(r); LP.append(logp)
        D.append(1.0 if done else 0.0); V.append(v)
        s = s2; ep_ret += r; gstep += 1
        if done:
            recent.append(ep_ret); recent = recent[-50:]; ep_ret = 0.0; s, _ = env.reset()
    with torch.no_grad():
        last_v = net.v(torch.from_numpy(s).float()).item()
    S = torch.tensor(np.array(S), dtype=torch.float); A = torch.tensor(np.array(A), dtype=torch.float)
    LP = torch.tensor(LP, dtype=torch.float)
    R = np.array(R, np.float32); D = np.array(D, np.float32); V = np.array(V, np.float32)
    adv = np.zeros(ROLLOUT, np.float32); gae = 0.0
    for t in reversed(range(ROLLOUT)):
        nv = last_v if t == ROLLOUT-1 else V[t+1]
        nonterminal = 1.0 - D[t]
        delta = R[t] + GAMMA*nv*nonterminal - V[t]
        gae = delta + GAMMA*LMBDA*nonterminal*gae; adv[t] = gae
    ret = torch.tensor(adv + V); adv = torch.tensor((adv - adv.mean())/(adv.std()+1e-8))
    idx = np.arange(ROLLOUT)
    for _ in range(K_EPOCH):
        np.random.shuffle(idx)
        for i in range(0, ROLLOUT, MINIBATCH):
            mb = idx[i:i+MINIBATCH]
            mu, std = net.pi(S[mb]); dist = Normal(mu, std)
            new_lp = dist.log_prob(A[mb]).sum(1)
            ratio = torch.exp(new_lp - LP[mb])
            surr1 = ratio*adv[mb]; surr2 = torch.clamp(ratio, 1-EPS_CLIP, 1+EPS_CLIP)*adv[mb]
            loss = -torch.min(surr1, surr2).mean() + 0.5*F.mse_loss(net.v(S[mb]), ret[mb]) \
                   - 0.01*dist.entropy().sum(1).mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 0.5); opt.step()
    mr = float(np.mean(recent)) if recent else float('nan')
    curve.append((gstep, mr)); print(f'step {gstep:7d}  mean50 {mr:7.1f}')
env.close()

## 4. The learning curve

In [ ]:
import matplotlib.pyplot as plt
xs, ys = zip(*curve)
plt.figure(figsize=(8,4.5))
plt.plot(xs, ys, color='#7c3aed', lw=2.2)
plt.axhline(200, color='#16a34a', ls='--', lw=1.4, label='solved (+200)')
plt.axhline(0, color='#9ca3af', lw=1)
plt.xlabel('environment steps'); plt.ylabel('mean return (last 50 episodes)')
plt.title('Continuous PPO on LunarLander'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Where to go next

- **Watch it land.** With `render_mode='human'` (locally) or by saving frames, roll out the
  trained `net` greedily (use `mu` without sampling) and watch the descent.
- **tanh-squash the policy.** Instead of clipping, map the Gaussian sample through `tanh` so
  actions are always in range — the trick SAC (a later topic) is built on.